# DalMax — Google Colab Pro runbook (notebook form)

Runnable companion to [`COLAB_RUNBOOK.md`](https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python/blob/main/COLAB_RUNBOOK.md) — every cell below
mirrors a numbered section of that runbook. **`COLAB_RUNBOOK.md` is the
source of truth**; if this notebook and the runbook ever disagree, trust the
runbook (and re-sync this notebook).

**Never run this notebook on the local no-GPU dev notebook**
(`.claude/rules/data-safety.md`) — it is for Google Colab Pro only.

Run every cell top to bottom, in order. The **Parameters** cell right below
is the only one you should normally need to edit.

In [ ]:
import os

# Parameters -- the ONLY cell in this notebook you should need to edit.
DRIVE_ROOT = "/content/drive/MyDrive/UFMS/Pós-graduação/Doutorado/FINAL/PROJETO/DALMAX"
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"
PART = "paper1"  # which campaign part the run cell executes: all | paper1 | upper_bound | rnhal | texhal
                 # (comma-separated allowed, e.g. "rnhal,texhal").
                 # Recommended session order (job counts from `python -m dalmax.campaign list`):
                 #   1) "paper1"      117 runs
                 #   2) "rnhal"        42 runs
                 #   3) "texhal"       30 runs
                 #   4) "upper_bound"   3 runs
                 #   (all = 192 runs total)
                 # Re-running the same PART resumes: completed runs are skipped, so a
                 # disconnect loses at most the run in progress.

# Also export as env vars so every `!`-shell cell below (and
# scripts/colab/setup_colab.sh's own DRIVE_ROOT override) sees them too.
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["PART"] = PART

print("DRIVE_ROOT:", DRIVE_ROOT)
print("REPO_URL:  ", REPO_URL)
print("REPO_DIR:  ", REPO_DIR)
print("BRANCH:    ", BRANCH)
print("PART:      ", PART)

## 0. Runtime check

See `COLAB_RUNBOOK.md` section 0. Select a GPU runtime (any); the cell below lists whichever GPU Colab assigned this session.
Its name is recorded automatically in every run's `run_metadata.json` (`environment.gpus`).

In [ ]:
!nvidia-smi

In [ ]:
import sys

# Informational only: Colab's SYSTEM Python is 3.13 (as of 2026-09-29), which is
# expected and fine. The project runs in a Poetry venv built on a uv-managed
# Python 3.12 (section 3); that venv interpreter is validated right after install.
print(f"Kernel (system) Python: {sys.version.split()[0]} -- any version is OK here")

## 1. Mount Google Drive

See `COLAB_RUNBOOK.md` section 1.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
assert os.path.isdir(f"{DRIVE_ROOT}/DATA/daninhas_full"), (
    f"{DRIVE_ROOT}/DATA/daninhas_full not found. Check DRIVE_ROOT in the "
    "Parameters cell above, and that daninhas_full was uploaded to Drive at "
    "that path -- every later step assumes it (COLAB_RUNBOOK.md section 1)."
)
print("Dataset folder found on Drive.")

In [ ]:
!ls "$DRIVE_ROOT/DATA/daninhas_full" | head

## 2. Clone or pull the repo

Idempotent: clones fresh into `/content/dalmax` (local runtime disk, not
Drive -- see `COLAB_RUNBOOK.md`'s hybrid-layout decision) the first time, or
pulls `BRANCH` if this runtime already has it (rare -- Colab runtimes are
usually fresh each session, but harmless to check). See
`COLAB_RUNBOOK.md` section 2.

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling latest {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
    )

# os.chdir affects `!`-cells too (they share the process cwd), but `%cd` is
# the documented/safe way to also update Colab's own displayed cwd state.
os.chdir(REPO_DIR)
%cd $REPO_DIR
!git log --oneline -1

## 3. Install dependencies with Poetry

See `COLAB_RUNBOOK.md` section 3.

In [ ]:
# Colab's system Python is 3.13, which torch==2.5.0 has no wheels for.
# Install a separate Python 3.12 with uv and point Poetry at it (idempotent).
!pip install -q uv poetry
!uv python install 3.12

In [ ]:
# Run from the repo root (REPO_DIR) so Poetry picks up pyproject.toml.
%cd {REPO_DIR}
!poetry env use "$(uv python find 3.12)"
!poetry install

This is a fresh runtime each session, so `poetry install` re-downloads the
pinned `torch==2.5.0`/`torchvision==0.20.0` CUDA wheels every time -- expect
roughly **5 minutes** (not previously measured on Colab specifically; treat
as an estimate). It creates an in-project `.venv/` (built on the uv-managed Python 3.12) on the local runtime disk,
same as the lab machine and local notebook.

**Alternative (not recommended by default)**:
`poetry config virtualenvs.create false --local` installs into the runtime's
system Python instead -- slightly faster, but downgrades Colab's preinstalled
torch build to the pinned 2.5.0 and loses environment isolation. See
`COLAB_RUNBOOK.md` section 3 for the full trade-off.

The SYSTEM Python on Colab is 3.13 (expected); it is only used to run `uv`/`poetry`.
The check cell below validates the venv interpreter instead.

In [ ]:
import subprocess

# Validate the Poetry venv interpreter (not the kernel's own Python).
probe = subprocess.run(
    ["poetry", "run", "python", "-c",
     "import sys, torch; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True, cwd=REPO_DIR,
)
print(probe.stdout.strip(), probe.stderr.strip()[-500:])
venv_version = probe.stdout.split()[0] if probe.returncode == 0 and probe.stdout.split() else ""
if not venv_version.startswith("3.12"):
    raise RuntimeError(
        f"Poetry venv Python is {venv_version or 'unavailable'}, expected 3.12.x "
        "(supported range 3.10-3.12; torch==2.5.0 has no 3.13 wheels). "
        "Re-run the previous cell: `uv python install 3.12`, then "
        '`poetry env use "$(uv python find 3.12)"` and `poetry install`. '
        "See COLAB_RUNBOOK.md section 7."
    )
print(f"Poetry venv Python {venv_version} OK")

In [ ]:
!poetry run python -c "import torch; print(torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"

## 4. Wire up the dataset and results/ (Drive)

`make colab-setup` runs `scripts/colab/setup_colab.sh`: it verifies Drive is
mounted, builds `daninhas_full.zip` on Drive on the **first session only**
(later sessions reuse it), unzips it onto the local runtime disk for fast
reads, and symlinks `results/` to Drive so every artifact survives a
disconnect. Idempotent -- safe to re-run after a disconnect. See
`COLAB_RUNBOOK.md` section 4.

In [ ]:
!make colab-setup

## 5. Sanity checks

See `COLAB_RUNBOOK.md` section 5.

In [ ]:
!make test

In [ ]:
!make smoke

In [ ]:
!make colab-check

### GPU identity check

Confirm this is the GPU you intended **before launching the campaign**. The cell below reads the
`run_metadata.json` written by the check run above (`environment` and `determinism` blocks).

In [ ]:
import glob
import json

hits = sorted(glob.glob("results/colab_check/**/run_metadata.json", recursive=True))
assert hits, "no run_metadata.json under results/colab_check/ -- did `make colab-check` finish?"
meta = json.load(open(hits[0]))
env = meta["environment"]
gpus = env.get("gpus") or []
assert gpus, "no GPU recorded in run_metadata.json -- is this a GPU runtime?"
gpu = gpus[0]
print("GPU name:      ", gpu.get("name"))
print("GPU memory GB: ", gpu.get("total_memory_gb"))
print("driver:        ", gpu.get("driver_version"))
print("CUDA (torch):  ", env["torch"].get("cuda"), "| cuDNN:", env["torch"].get("cudnn"))
print("determinism:   ", meta["determinism"])

In [ ]:
!poetry run python tools/loader.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth

In [ ]:
import glob

img = sorted(glob.glob("DATA/daninhas_full/test/DATASET_GRAMINEA/*"))[0]
print(img)
!poetry run python tools/predict.py --model results/colab_check/daninhas_full/SEED_1/NQ_100_NIL_100_NR_1_NE_10/SSRAEKmeansHCSampling/saved_model.pth --image "{img}"

**Only continue to section 6 once all of the above pass.**

## 6. The campaign -- the single path for the final execution

One manifest drives the single, deduplicated execution of everything the three papers need
(paper 1: 12 classical strategies + KMH x nq {10,50,100} + the `FullSupervised` upper bound;
paper 2/3: TexHAL/RNHAL ablations at nq=100): **192 runs / 64 run groups**. See `COLAB_RUNBOOK.md`
"The campaign" and `.specs/experiments/campaign.md`. The `PART` parameter selects `all` or a subset
(one part per session, see below). Budget Drive space for ~192 checkpoints (~18 GB).

In [ ]:
!make campaign-list PART={PART}

### Splitting the campaign across sessions

The full campaign (192 runs) does not fit in one Colab session. Run one `PART` per session (edit
`PART` in the Parameters cell, re-run cells 0-4, then the run cell):

| Session | `PART` | Runs |
|---|---|---|
| 1 | `paper1` | 117 |
| 2 | `rnhal` | 42 |
| 3 | `texhal` | 30 |
| 4 | `upper_bound` | 3 |

Every run is skipped if its results already exist, so re-running the same `PART` after a disconnect or
session limit **resumes** where it stopped (at most the run in progress is lost). Use the Progress cell
below at any time to see how far along a part is.

Run (skip-existing: **re-running this cell resumes after a disconnect** -- redo sections 0-4 first).
The GPU identity was already confirmed by the check cell in section 5.


In [ ]:
import time

start = time.time()
!make campaign-run PART={PART}

In [ ]:
elapsed_hours = (time.time() - start) / 3600
print(f"Campaign part(s) {PART} took {elapsed_hours:.2f} hours on this runtime.")
print("Paste this number into COLAB_RUNBOOK.md's campaign section (wall-clock TBD).")

**Progress** (usable anytime, e.g. from a second cell while the run is going, or after a disconnect):

In [ ]:
!make campaign-verify PART={PART} 2>&1 | tail -n 15
!tail -n 20 results/campaign/campaign.log 2>/dev/null || echo "no campaign.log yet"

In [ ]:
!cat results/campaign/failures.log 2>/dev/null || echo "no failures logged"

**Verify**: OK/INCOMPLETE/MISSING per job plus the seed-consistency audit (every run of a seed must start
from the identical initial labeled set). Exit code 0 = all OK, 1 = incomplete/missing job or FAIL audit,
2 = only audit WARNs. Re-run the run cell above to redo failed/missing jobs.

In [ ]:
!make campaign-verify PART={PART}

**Report** + copy to Drive (tables md/tex/csv, `seed_audit.md`, mean confusion matrices).

In [ ]:
!make campaign-report

In [ ]:
!mkdir -p "$DRIVE_ROOT/results"
!cp -r docs/results/campaign "$DRIVE_ROOT/results/campaign_tables"

In [ ]:
!ls "$DRIVE_ROOT/results/campaign_tables"

## 7. Troubleshooting

See `COLAB_RUNBOOK.md` section 7 for full detail. Quick pointers:

- Drive FUSE slowness / `Transport endpoint is not connected` -> remount
  Drive (section 1) and re-run `!make colab-setup`.
- `results/` symlink or local `results/` conflicts -> read
  `scripts/colab/setup_colab.sh`'s error message; it names the exact
  conflicting path.
- Out of GPU memory -> copy the params JSON into gitignored `files_config/local/`
  and lower `batch_size` there; never edit the committed configs in place.
- Session limit reached -> re-run sections 0-4 in a fresh runtime, then the
  run cell again (skip-existing resumes; only complete leaves are skipped).
- Python 3.13 system runtime (expected on Colab) -> handled by the uv-managed 3.12
  in section 3; see `COLAB_RUNBOOK.md` section 7 if the venv check fails.